# Reproducible scripts for Capstone C: Hydrogen Value-Chain Integration

This companion notebook was generated from fenced `python` code blocks in `chapters/ch26_capstone_value_chain/chapter.md`. Blocks marked `noexec` in the chapter are preserved for reading but are not executed here.


In [1]:
from pathlib import Path
import re
import shutil

try:
    from IPython.display import Image, display
except Exception:
    Image = None
    display = None

_CHAPTER_FIGURES_DIR = Path("../figures")
_CHAPTER_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
_PAPERLAB_PATTERNS = ("*.png", "*.jpg", "*.jpeg", "*.svg", "*.webp", "*.csv", "*.json", "*.txt", "*.html")
_PAPERLAB_IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg"}


def _paperlab_stamp(path):
    try:
        stat = path.stat()
        return (stat.st_mtime_ns, stat.st_size)
    except OSError:
        return None


def _paperlab_scan_files():
    files = {}
    roots = [Path("."), _CHAPTER_FIGURES_DIR]
    for root in roots:
        if not root.exists():
            continue
        for pattern in _PAPERLAB_PATTERNS:
            for path in root.glob(pattern):
                if path.is_file():
                    files[path.resolve()] = _paperlab_stamp(path)
    return files


def _paperlab_safe_name(name):
    cleaned = re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("._")
    return cleaned or "notebook_output"


_paperlab_seen_files = _paperlab_scan_files()


def _paperlab_capture_new_files(label):
    global _paperlab_seen_files
    current = _paperlab_scan_files()
    changed = []
    for path, stamp in current.items():
        if _paperlab_seen_files.get(path) != stamp:
            changed.append(Path(path))
    _paperlab_seen_files = current

    for path in sorted(changed):
        suffix = path.suffix.lower()
        if suffix in _PAPERLAB_IMAGE_SUFFIXES:
            if path.parent.resolve() == _CHAPTER_FIGURES_DIR.resolve():
                dest = path
            else:
                dest = _CHAPTER_FIGURES_DIR / _paperlab_safe_name(f"{label}_{path.name}")
                shutil.copy2(path, dest)
            print(f"Captured figure: figures/{dest.name}")
            if Image is not None and display is not None:
                display(Image(filename=str(dest)))
        else:
            print(f"Generated result file: {path}")

## Example 1 from `chapters/ch26_capstone_value_chain/chapter.md` line 116


This block is preserved but not executed because it is noexec marker.

````python
from neqsim import jneqsim as J

# Direct Java access through neqsim-python. Use explicit units and call
# setMixingRule before running flashes or process equipment.
# Multi-area ProcessModel: production + conditioning + transport, then economics.
plant = J.process.processmodel.ProcessModel()

# --- Production area: blue SMR with capture ---
prod = J.process.processmodel.ProcessSystem()
builder = J.process.hydrogen.BlueHydrogenPlantBuilder()
builder.setName("Value-chain blue H2")
builder.setMethaneFeedMolePerSec(120.0)
builder.setSteamToCarbonRatio(3.0)
builder.setCo2CaptureFraction(0.92)
builder.setH2ExportPressure(70.0)
builder.setIncludePsa(True)
prod = builder.build()
plant.add("Production", prod)

# --- Transport area: 500 km pipeline at 70 bar ---
trans = J.process.processmodel.ProcessSystem()
h2_product = builder.getHydrogenProductStream()
pipe = J.process.equipment.pipeline.PipeBeggsAndBrills("500 km H2 pipeline", h2_product)
pipe.setLength(500000.0)
pipe.setDiameter(1.0)
pipe.setNumberOfIncrements(50)
pipe.setConstantSurfaceTemperature(5.0, "C")
trans.add(pipe)
plant.add("Transport", trans)

plant.run()

# --- Economics: levelized cost of delivered H2 ---
dcf = J.process.util.fielddevelopment.DCFCalculator()
dcf.setDiscountRate(0.08)
dcf.setProjectLife(25)
annual_h2_kg = builder.getHydrogenProductMassFlowKgPerHour() * 8000.0
capex_USD = 1500.0e6   # production + pipe
opex_USD_per_yr = 60.0e6
gas_USD_per_GJ = 5.0

print({
    "h2_kg_per_yr": annual_h2_kg,
    "pipeline_outlet_P_bara": pipe.getOutletStream().getPressure("bara"),
    "carbon_intensity_kgCO2_per_kgH2": builder.getCarbonIntensityKgCO2PerKgH2(),
    "indicative_LCOH_USD_per_kgH2": (
        (capex_USD * 0.08 / (1.0 - (1.08) ** -25)) + opex_USD_per_yr
    ) / annual_h2_kg,
})

````
